<a href="https://colab.research.google.com/github/zenight01/googlecolab/blob/main/Ligaya_DianneChristel_Lab08_ALGOCOM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **LABORATORY ACTIVITY 8: Neural Network Algorithms**

**Name:** LIGAYA, Dianne Christel L.

**Section:** III-CCSAD

**Date:** 10/22/2026

---

## **📋 Instructions & Steps**
1. **Save a copy** of this notebook to your personal Google Drive or local workspace before running or modifying any code.
2. **Individual Activity:** This laboratory exercise is to be completed individually.
3. **Accomplish the Core Tasks:** Review and run the provided Hands-On Lab Demonstrations, then proceed to complete the *Student Exercises & Lab Deliverables* and *Review Questions*.
4. **Self-Review:** Before submitting, cross-check your work against the Laboratory Performance Rubric at the end of this document to ensure all requirements are met.

---

## **📤 Submission Guidelines**
* **Deliverable:** Submit your completed `.ipynb` file (or exported PDF with visible code and execution outputs) containing all written responses and completed coding exercises.
* **File Naming Convention:** `[LastName]_[FirstName]_Lab08_ALGOCOM.ipynb` (or `.pdf`)
* **Submission Policy:** Submit your individual work through the designated learning management system (LMS) strictly on or before the allotted deadline.

---

## **🟡 AI Policy: Yellow Zone**
* **Allowed:** You may use AI tools (such as ChatGPT, Gemini, or GitHub Copilot) to help explain tensor operations, clarify backpropagation mechanics, suggest Keras layer syntax, or brainstorm network topologies for your coding scenarios.
* **Required:** If you use AI to assist with your code, you **must document it** within a comment block in the relevant section.
  * *Example:* `# AI Assistance: I used Gemini to help define the Conv1D layer input shape and configure the MaxPooling1D parameters for Scenario 3.`
* **Expectation:** Ensure that all outputs are critically reviewed and understood. Do not blindly copy-paste code. You are responsible for ensuring your neural networks train properly, avoid data leakage, and achieve valid convergence metrics.

## **🧪 Part 1: Hands-On Lab Demonstrations**
**Objective:** Construct, train, regularize, and evaluate an end-to-end Multi-Layer Perceptron (MLP) for multi-class animal welfare triage using TensorFlow and Keras.

### **Demonstration 1: Telemetry Ingestion, Imputation & Feature Scaling**

In [ ]:
# ==============================================================================
# DEMO 1: Ingestion, Missing Data Handling & Scaling
# ==============================================================================
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical

np.random.seed(42)

# Generate 2,500 synthetic animal shelter intake records across 6 features
# Classes: 0 = At-Risk (20%), 1 = Standard Adoption (55%), 2 = High-Velocity Urgent (25%)
X_raw, y_raw = make_classification(
    n_samples=2500,
    n_features=6,
    n_informative=5,
    n_redundant=1,
    n_classes=3,
    weights=[0.20, 0.55, 0.25],
    flip_y=0.02,
    random_state=42
)

feature_names = [
    'age_months', 'health_score', 'social_affinity',
    'activity_hours', 'human_responsiveness', 'environmental_stress'
]

df_shelter = pd.DataFrame(X_raw, columns=feature_names)
df_shelter['welfare_class'] = y_raw

# Inject realistic missing values into rescue health & social records
df_shelter.loc[np.random.choice(df_shelter.index, 60, replace=False), 'health_score'] = np.nan
df_shelter.loc[np.random.choice(df_shelter.index, 45, replace=False), 'social_affinity'] = np.nan

# Separate features and target
X = df_shelter.drop(columns=['welfare_class'])
y = df_shelter['welfare_class'].values

# Split data (80% Train, 20% Held-Out Test) maintaining class proportions
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# Impute missing entries using training medians to prevent leakage
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train_raw)
X_test_imputed = imputer.transform(X_test_raw)

# Standardize feature scales for gradient descent optimization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

# One-hot encode targets for categorical cross-entropy loss
y_train_cat = to_categorical(y_train_raw, num_classes=3)
y_test_cat = to_categorical(y_test_raw, num_classes=3)

print("Ingestion Pipeline Complete.")
print(f"Training Features Tensor Shape : {X_train_scaled.shape}")
print(f"Training Targets One-Hot Shape : {y_train_cat.shape}")

Ingestion Pipeline Complete.
Training Features Tensor Shape : (2000, 6)
Training Targets One-Hot Shape : (2000, 3)


**Expected Output:**
```text
Ingestion Pipeline Complete.
Training Features Tensor Shape : (2000, 6)
Training Targets One-Hot Shape : (2000, 3)
```

---

### **Demonstration 2: Deep MLP Assembly with Batch Normalization & Dropout**

In [ ]:
# ==============================================================================
# DEMO 2: Regularized Deep Multi-Layer Perceptron Assembly
# ==============================================================================
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

tf.random.set_seed(42)

# Build deep Multi-Layer Perceptron
triage_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],), name='Hidden_Layer_1'),
    BatchNormalization(name='BatchNorm_1'),
    Dropout(0.30, name='Dropout_1'),

    Dense(64, activation='relu', name='Hidden_Layer_2'),
    BatchNormalization(name='BatchNorm_2'),
    Dropout(0.20, name='Dropout_2'),

    Dense(32, activation='relu', name='Hidden_Layer_3'),
    Dense(3, activation='softmax', name='Output_Layer')
])

triage_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

triage_model.summary()

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


**Expected Output:**
```text
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
=================================================================
 Hidden_Layer_1 (Dense)      (None, 128)               896       
 BatchNorm_1 (BatchNorma    (None, 128)               512       
 Dropout_1 (Dropout)         (None, 128)               0         
 Hidden_Layer_2 (Dense)      (None, 64)                8256      
 BatchNorm_2 (BatchNorma    (None, 64)                256       
 Dropout_2 (Dropout)         (None, 64)                0         
 Hidden_Layer_3 (Dense)      (None, 32)                2080      
 Output_Layer (Dense)        (None, 3)                 99        
=================================================================
Total params: 12,099 (47.26 KB)
Trainable params: 11,715 (45.76 KB)
Non-trainable params: 384 (1.50 KB)
_________________________________________________________________
```

---

### **Demonstration 3: Adaptive Training with Early Stopping & LR Scheduling**

In [ ]:
# ==============================================================================
# DEMO 3: Training with EarlyStopping and ReduceLROnPlateau Callbacks
# ==============================================================================
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=0.0001,
    verbose=1
)

history = triage_model.fit(
    X_train_scaled,
    y_train_cat,
    validation_split=0.20,
    epochs=40,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=0
)

print(f"Training completed successfully across {len(history.history['loss'])} epochs.")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.4f}")
print(f"Final Validation Accuracy: {history.history['val_accuracy'][-1]*100:.2f}%")

**Expected Output:**
```text
Epoch 15: ReduceLROnPlateau reducing learning rate to 0.002500000037252903.
Restoring model weights from the end of the best epoch: 22.
Epoch 30: early stopping
Training completed successfully across 30 epochs.
Final Validation Loss: 0.3912
Final Validation Accuracy: 84.00%
```

---

### **Demonstration 4: Model Evaluation Diagnostics & Feature Attribution**

In [ ]:
# ==============================================================================
# DEMO 4: Held-Out Diagnostics & Permutation Importance
# ==============================================================================
from sklearn.metrics import classification_report, confusion_matrix, f1_score

test_loss, test_acc = triage_model.evaluate(X_test_scaled, y_test_cat, verbose=0)
y_pred_probs = triage_model.predict(X_test_scaled, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)

target_labels = ['At-Risk (0)', 'Standard (1)', 'High-Velocity (2)']

print("=== HELD-OUT INTAKE RECORD EVALUATION ===")
print(f"Test Loss: {test_loss:.4f} | Test Accuracy: {test_acc*100:.2f}%\n")
print(classification_report(y_test_raw, y_pred, target_names=target_labels))

# Compute permutation feature attribution
baseline_f1 = f1_score(y_test_raw, y_pred, average='macro')
importance_scores = []

for col_idx in range(X_test_scaled.shape[1]):
    X_test_permuted = X_test_scaled.copy()
    np.random.shuffle(X_test_permuted[:, col_idx])
    perm_preds = np.argmax(triage_model.predict(X_test_permuted, verbose=0), axis=1)
    importance_scores.append(baseline_f1 - f1_score(y_test_raw, perm_preds, average='macro'))

print("=== INTAKE TELEMETRY IMPORTANCE RANKINGS ===")
for rank, idx in enumerate(np.argsort(importance_scores)[::-1], 1):
    print(f"{rank}. {feature_names[idx]:22} | Impact on Macro F1: {importance_scores[idx]:.4f}")

**Expected Output:**
```text
=== HELD-OUT INTAKE RECORD EVALUATION ===
Test Loss: 0.3891 | Test Accuracy: 84.80%

                   precision    recall  f1-score   support

      At-Risk (0)       0.84      0.80      0.82       102
     Standard (1)       0.86      0.88      0.87       271
High-Velocity (2)       0.82      0.81      0.82       127

         accuracy                           0.85       500
        macro avg       0.84      0.83      0.84       500
     weighted avg       0.85      0.85      0.85       500

=== INTAKE TELEMETRY IMPORTANCE RANKINGS ===
1. health_score           | Impact on Macro F1: 0.2215
2. social_affinity        | Impact on Macro F1: 0.1642
3. age_months             | Impact on Macro F1: 0.0984
4. human_responsiveness   | Impact on Macro F1: 0.0412
5. environmental_stress   | Impact on Macro F1: 0.0195
6. activity_hours         | Impact on Macro F1: 0.0081
```

## **📝 Part 2: Student Exercises & Lab Deliverables**

### **Conceptual Questions**
1. **Vanishing Gradient Mechanics:** Why does the Sigmoid activation function trigger the vanishing gradient problem in deep networks, and how does the Rectified Linear Unit (ReLU) mathematically eliminate this limitation for positive activations?
   - **Answer:** Sigmoid turns any input into a value between 0 and 1. However, when the input is very large or very small, its output barely changes, so its gradient becomes close to zero. During training, the network passes gradients backward through each layer. If these gradients keep getting smaller, the earlier layers receive very little information about how to improve. This makes learning slow, especially in deep networks.

ReLU works differently. It returns 0 for negative inputs and keeps positive inputs unchanged. Its gradient is 1 for positive inputs, so it does not shrink the gradient as it passes through active neurons. This helps the network learn more effectively.

2. **Categorical Cross-Entropy vs. MSE:** In multi-class classification tasks using a Softmax output layer, why is Categorical Cross-Entropy mathematically and practically preferred over Mean Squared Error (MSE)? Focus on gradient magnitude when predictions are confident but wrong.
   - **Answer:** Categorical cross-entropy is commonly used for classification because it helps the model learn to predict the correct class. When used with softmax, its output gradient is simply the prediction minus the correct answer. If the model is very confident but predicts the wrong class, the error is large, so the model receives a strong signal to correct itself.

Mean Squared Error (MSE) works differently. When combined with softmax, its gradient includes an extra factor that can become very small when the model is highly confident. This means a model that confidently predicts the wrong class may receive a weak learning signal. As a result, it can take longer to correct its mistakes. Cross-entropy is generally a better choice for classification because it is designed to measure errors in predicted probabilities.

3. **Batch Normalization Dynamics:** How does standardizing internal activations via Batch Normalization reduce internal covariate shift, and why does this allow the model to use significantly higher learning rates during training?
   - **Answer:** Batch normalization adjusts the values passing through a layer so they have a more consistent average and spread. It also learns how to scale and shift those values when needed.

During training, the weights of earlier layers keep changing, which also changes the values received by later layers. This can make training harder because each layer has to keep adjusting to new inputs. Batch normalization helps control these changes and prevents values from becoming too large or too small.

Because of this, training can become faster and more stable. It can also allow the model to use a higher learning rate without training becoming unstable.

### **Applied Coding Scenarios**

#### **Scenario 1 — Binary Wildlife Camera-Trap Sorter (Cat vs. Dog)**
A wildlife sanctuary deploys motion-activated camera stations and requires an automated binary classifier to distinguish between wild **Felines (0)** and **Canines (1)** based on morphological traits: body length, shoulder height, ear aspect ratio, and snout-to-skull index.
1. Generate an imbalanced synthetic dataset of 1,200 animals using `make_classification(n_samples=1200, n_features=4, weights=[0.85, 0.15], random_state=42)`.
2. Construct and train two models using `binary_crossentropy`:
   * **Model A (Single-Neuron Perceptron):** `Dense(1, activation='sigmoid')`
   * **Model B (Non-Linear MLP):** `Dense(16, activation='relu')` $\rightarrow$ `Dense(1, activation='sigmoid')`
3. Print the Confusion Matrix and Recall score for both models on the held-out test split. Explain why the non-linear MLP is better equipped to identify the minority canine class.

In [ ]:
# Scenario 1: Binary Wildlife Camera-Trap Sorter Implementation
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, recall_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# TODO: Document AI assistance if used
# AI Assistance: [Document here]

# TODO: 1. Generate dataset and perform train-test split (80/20, stratify=y)

# TODO: 2. Scale features using StandardScaler

# TODO: 3. Assemble and compile Model A (Single-Neuron Perceptron)

# TODO: 4. Assemble and compile Model B (Non-Linear MLP)

# TODO: 5. Train both models on the training set

# TODO: 6. Predict on the test set and display Confusion Matrix and Recall for both


#### **Scenario 2 — Optimizer Convergence Comparison (SGD vs. RMSprop vs. Adam)**
An animal shelter analytics department wants to benchmark how different optimization algorithms traverse loss landscapes on complex, multi-feature intake data.
1. Synthesize 1,500 samples across 10 features with 3 target classes using `make_classification`.
2. Train three identical network topologies (`Dense(64, activation='relu')` $\rightarrow$ `Dense(32, activation='relu')` $\rightarrow$ `Dense(3, activation='softmax')`) for 30 epochs using:
   * **Model A:** `SGD(learning_rate=0.01)`
   * **Model B:** `RMSprop(learning_rate=0.001)`
   * **Model C:** `Adam(learning_rate=0.001)`
3. Plot the training loss curves for all three optimizers on a single comparative chart. Explain how Adam's combined tracking of momentum and second-order gradient moments achieves faster, smoother convergence.

In [ ]:
# Scenario 2: Optimizer Convergence Benchmark Implementation
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, RMSprop, Adam

# TODO: Document AI assistance if used
# AI Assistance: [Document here]

# TODO: 1. Generate multi-class dataset (1500 samples, 10 features, 3 classes)

# TODO: 2. Scale features and one-hot encode targets

# TODO: 3. Create helper function to return identical uncompiled MLP model architecture

# TODO: 4. Compile and train Model A (SGD), Model B (RMSprop), and Model C (Adam) for 30 epochs

# TODO: 5. Plot training loss histories on a single Matplotlib graph


#### **Scenario 3 — Smart Collar Telemetry Classification via 1D Convolutions**
A pet health startup processes continuous motion data from smart collars (3-axis accelerometer and heart rate telemetry over time) to classify pet behavioral states into **Resting (0)**, **Playing (1)**, or **Acute Pain/Distress (2)**.
1. Generate synthetic sequential telemetry of shape `(1000, 12, 1)` representing 12-second continuous sensor readings.
2. Construct a 1D Convolutional Neural Network consisting of:
   * `Conv1D(filters=16, kernel_size=3, activation='relu', input_shape=(12, 1))`
   * `BatchNormalization()`
   * `MaxPooling1D(pool_size=2)`
   * `Flatten()`
   * `Dense(32, activation='relu')`
   * `Dense(3, activation='softmax')`
3. Train the architecture and output a full `classification_report`. Explain why sliding convolutional kernels capture localized temporal patterns in wearable biometric data more effectively than dense fully connected layers.

In [ ]:
# Scenario 3: 1D Convolutional Sensor Classification Implementation
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, BatchNormalization, Flatten, Dense
from tensorflow.keras.utils import to_categorical

# TODO: Document AI assistance if used
# AI Assistance: [Document here]

# TODO: 1. Generate synthetic sequential data of shape (1000, 12, 1) and discrete labels (3 classes)

# TODO: 2. Train-test split (80/20) and one-hot encode targets

# TODO: 3. Assemble Conv1D architecture with BatchNorm, MaxPool1D, Flatten, and Dense layers

# TODO: 4. Compile with 'adam' and 'categorical_crossentropy' and train for 20 epochs

# TODO: 5. Evaluate and print classification report on test partition


#### **Scenario 4 — Real-World Modeling Challenge (Student's Choice)**
1. **Scenario Description:** Identify a prediction problem from your everyday life or another scientific domain (e.g., student course completion, bus transit arrival delays, streaming recommendation skips, or athletic injury risk). State the target variable, classification type (binary or multi-class), and define 4 to 6 relevant input features.
   - *(Type description here)*
2. **Architecture Selection & Justification:** Design an appropriate neural network topology (an MLP or 1D CNN). Justify your choices for layer count, activation functions, regularization (Dropout/Batch Normalization), and optimizer based on the size and complexity of your dataset.
   - *(Type justification here)*
3. **Python Implementation:** Write a complete, standalone script that loads or simulates the data, applies proper scaling without leakage, trains the model using Early Stopping, and outputs a Confusion Matrix alongside Macro $F_1$-score.

In [ ]:
# Scenario 4: Real-World Implementation (Student's Choice)
# TODO: Document AI assistance if used
# AI Assistance: [Document here]

# TODO: Write your end-to-end neural network pipeline here


## **🔍 Part 3: Review Questions & Self-Assessment Checkpoints**

1. **The XOR Dilemma:** Why is a single-layer perceptron mathematically incapable of learning the Exclusive-OR (XOR) function, and how does introducing a hidden layer with non-linear activations resolve this limitation geometrically?
   - **Answer:** A single-layer perceptron can only separate data using one straight line. XOR is a problem because its inputs cannot be separated into the correct classes using just one line.

For XOR, the inputs (0,0) and (1,1) belong to one class, while (0,1) and (1,0) belong to the other. No single straight line can separate these two groups correctly.

Adding a hidden layer with a non-linear activation allows the network to combine multiple boundaries. For example, hidden neurons can learn patterns similar to OR and NAND, while the output neuron combines their results using AND. This allows the network to solve XOR even though a single perceptron cannot.

2. **Dying ReLU Problem:** Under what specific training conditions does a neuron using the standard ReLU activation function "die," and how does Leaky ReLU or GELU prevent this loss of network capacity?
   - **Answer:** A ReLU neuron dies when its input stays negative for every training example. This can happen when a large weight update pushes its weights or bias too far in the wrong direction.

When this happens, the neuron always outputs 0 and receives a gradient of 0. Since its weights no longer get updated, it may stop learning completely. This also means fewer neurons are available to learn useful patterns.

Leaky ReLU helps prevent this by allowing a small negative output when the input is below zero. For example, it can return \(0.01x\) for negative inputs. This allows a small gradient to pass through, giving the neuron a chance to recover.

GELU is another activation function that changes smoothly instead of cutting off all negative inputs. It allows small negative values and gradients to pass through, which can also help during training.

3. **Dropout Mechanics Across Modes:** Explain the operational difference in how Dropout functions during the **training phase** versus the **inference (evaluation) phase**. Why must neuron activations be scaled during testing if Dropout was applied during training?
   - **Answer:** Dropout randomly turns off some neurons during training. This prevents the network from depending too much on specific neurons and encourages it to learn patterns using different combinations of neurons.

During inference, which means using the trained model to make predictions, dropout is turned off and all neurons are active.

However, using more active neurons during inference could increase the values passed to the next layer compared to training. To avoid this problem, Keras uses a method called inverted dropout. It increases the values of the neurons that remain active during training by multiplying them by \(1/(1-p)\), where \(p\) is the dropout rate.

This keeps the values at a similar scale, so no extra scaling is needed during inference.

4. **Backpropagation & The Chain Rule:** During backpropagation, how does the calculus chain rule compute error gradients for the earliest layers in a deep neural network, and why does this process cause gradients to vanish when using saturating activations?
   - **Answer:** Backpropagation calculates how much each weight contributes to the model's error. It uses the chain rule to pass this information backward through the network.

To calculate the gradient for an early layer, the network multiplies the gradients and derivatives of the layers between that layer and the final error. If these values are repeatedly small, the gradient becomes smaller as it travels backward.

This is what happens with sigmoid in deep networks. Its derivative is never greater than 0.25 and can become even closer to zero for very large or small inputs. Multiplying these small values across many layers causes the gradient to shrink rapidly.

As a result, the early layers receive very little information about how to adjust their weights. This is called the vanishing gradient problem, and it can make deep networks difficult to train.

5. **Edge Deployment Constraints:** Why are deep floating-point (`float32`) neural networks quantized into 8-bit integers (`int8`) before deployment onto embedded edge hardware (such as smart pet collars or camera traps), and what trade-offs occur regarding model size, latency, and predictive accuracy?
   - **Answer:** Devices such as animal collars and camera traps often have limited memory, processing power, and battery life. A large model can be difficult to run on these devices.

INT8 quantization reduces the model's size by representing its weights using 8-bit integers instead of 32-bit floating-point numbers. This can make the model's weights roughly four times smaller. It can also speed up predictions and reduce power use, especially on devices with hardware designed for integer calculations.

The main drawback is that INT8 cannot represent values as precisely as float32. Converting values to a smaller range of numbers introduces rounding errors, which can reduce the model's accuracy.

To reduce this loss, developers can use calibration data to choose suitable value ranges or apply quantization-aware training, where the model is trained while accounting for the effects of quantization.

## **📊 Part 4: Laboratory Performance Rubric**

| Evaluation Criteria | Unsatisfactory (<60%) | Needs Improvement (60–69%) | Satisfactory (70–79%) | Good (80–89%) | Excellent (90–100%) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Model Implementation & Review Qs (20%)** | Script throws execution errors; fails to instantiate Keras layers; Review Questions are largely blank. | Builds models, but hardcodes incorrect activations or loss functions; answers show weak theoretical grasp. | Correctly implements MLPs with valid activations and loss functions; adequately answers Review Questions. | Clean implementation of feedforward topologies and convolutional layers; provides clear, accurate theoretical answers. | Flawless model implementation using custom topologies, proper tensor formatting, and advanced callbacks; demonstrates deep theoretical mastery in Review Questions. |
| **Preprocessing & Pipeline Integrity (20%)** | Causes severe data leakage; fails to handle missing values or skips feature standardization. | Imputes data naively (e.g., zero-fill); applies transformations inconsistently across splits. | Prepares data adequately; applies median imputation and standardizes feature scales on training data. | Clean, leak-free preprocessing workflow using `StandardScaler` and `SimpleImputer` across separate folds. | Enterprise-grade preprocessing pipeline; ensures complete data isolation across splits with automated one-hot encoding and robust tensor reshaping. |
| **Optimization & Regularization (20%)** | Fails to train models; omits optimizers or suffers from extreme numerical overflow/divergence. | Models train, but suffer from heavy overfitting or severe gradient saturation without regularization. | Successfully uses Adam/SGD optimizers; applies basic Dropout to mitigate overfitting. | Effectively tunes learning rates; uses Batch Normalization and Early Stopping to stabilize convergence. | Expertly integrates Adam/RMSprop, dynamic learning rate schedules (`ReduceLROnPlateau`), Batch Normalization, and Dropout to optimize the loss landscape. |
| **Scenario Logic & Code Execution (20%)** | No functional code provided for Scenarios 1–4. | Implements scenarios superficially; uses incorrect architectures or unsuitable evaluation metrics. | Functional code for all scenarios; computes basic accuracy and generates confusion matrices. | Well-structured implementations across all scenarios; properly interprets classification reports and comparative loss curves. | Outstanding scenario implementations; thoroughly justifies architectural choices in Scenario 4 and implements clean 1D CNN and optimizer benchmarks. |
| **Code Cleanliness & Documentation (20%)** | Disorganized script; no comments; undocumented AI usage. | Messy formatting; sparse comments; describes metrics mechanically without contextual meaning. | Readable, PEP-8 compliant code; includes basic performance summaries and required AI documentation. | Well-commented, modular code; clearly connects model outputs, confusion matrices, and feature attributions to domain problems. | Production-quality code structure; clear documentation; provides detailed analytical commentary linking training dynamics to underlying machine learning theory. |